# Latent Failure Forecasting PoC on Google Colab (Live Sandbox)

**Session crashed (RAM/OOM)?** `Runtime → Factory reset session`, rerun from top.

**Colab free tier (~12 GB system RAM) cannot reliably load 3B** — TransformerLens briefly holds two weight copies in CPU RAM during conversion. The pipeline **auto-falls back to 1B** unless you set `VERITAS_FORCE_MODEL=1`.

| Goal | Settings |
|------|----------|
| **Works on free T4** (default) | 1B auto — no env needed |
| **Force 3B** (may crash) | `VERITAS_FORCE_MODEL=1` before load |
| **8B** | A100/L4 only — not T4 |

Default: **`meta-llama/Llama-3.2-1B-Instruct`** on Colab, **`n_ctx=8192`**.

1. T4 GPU → **Restart session** after install
2. clone → install → **restart** → GPU check → verify deps → HF login → **pipeline** (skip smoke cell unless debugging load)
3. Never `reload(torch)` after pip

**Smoke:** `os.environ['VERITAS_SMOKE_N']='2'` before pipeline.

Delete old artifacts when switching models: `results/trajectories.json`, `data/activations/*.pt`.

In [ ]:
import os
if os.path.basename(os.getcwd()) != 'algoverse-PoC':
    if not os.path.isdir('algoverse-PoC'):
        !git clone https://github.com/abdelmagid07/algoverse-PoC.git
    %cd algoverse-PoC
!git pull --ff-only

In [ ]:
# Install WITHOUT importing torch in this kernel (importing torch before pip,
# then reinstalling/reloading it, corrupts the runtime). Read Colab torch in a
# subprocess only. Install TL with --no-deps so pip does not touch torch.
import subprocess, sys

def _torch_tuple(ver: str) -> tuple[int, ...]:
    return tuple(int(x) for x in ver.split('+')[0].split('.')[:3])

out = subprocess.check_output([
    sys.executable, '-c',
    'import torch; print(torch.__version__); print(torch.version.cuda or "128")',
], text=True).strip().splitlines()
colab_torch, cuda = out[0], out[1]
if _torch_tuple(colab_torch) < (2, 7, 0):
    raise RuntimeError(f'Colab torch {colab_torch} < 2.7; factory-reset runtime.')
print(f'Colab torch {colab_torch} (subprocess check — kernel has not imported torch)')

# TL deps except torch/torchvision (pip must not replace Colab's CUDA torch).
_TL_DEPS = (
    'accelerate beartype better-abc einops fancy-einsum jaxtyping rich '
    'sentencepiece transformers-stream-generator typeguard typing-extensions '
    'tqdm huggingface-hub packaging protobuf pandas wandb'
)

# Remove Colab's torchvision first (we never use vision; mismatched builds crash
# transformers). TL lists torchvision as a dep but only imports it in try/except.
!pip uninstall -q -y torchvision || true

!pip install -q --no-deps "transformer_lens>=3.4.0"
!pip install -q "transformers>=5.4.0"
!pip install -q {_TL_DEPS}
!pip install -q datasets scipy scikit-learn matplotlib numpy
!pip uninstall -q -y torchvision || true

print('\nInstall done. (Ignore torchvision version warnings — it is removed above.)')
print('Do NOT import torch/transformers here.')
print('>>> Runtime > Restart session, then run GPU-check + Verify-deps cells.')

In [ ]:
import torch
assert torch.cuda.is_available(), (
    'CUDA not visible. Set runtime to T4 GPU, then Runtime > Restart session, then re-run.'
)
print(torch.cuda.get_device_name(0))
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
import psutil
print(f'System RAM: {psutil.virtual_memory().total / 1e9:.1f} GB')
print(f'torch={torch.__version__}')

In [ ]:
# Run AFTER restart — first transformers import in this session.
import inspect
import torch
import transformers
import transformer_lens

if 'reason' not in inspect.signature(torch.compiler.disable).parameters:
    raise RuntimeError(
        f'torch {torch.__version__} missing compiler.disable(reason=). '
        f'Factory-reset runtime, run clone+install (no GPU check before install), restart, retry.'
    )
print(f'torch={torch.__version__}  transformers={transformers.__version__}  '
      f'transformer_lens={getattr(transformer_lens, "__version__", "?")}')
print('Imports OK — continue to HF login.')

In [ ]:
from getpass import getpass
from huggingface_hub import login
login(getpass('HF token: '))

In [ ]:
# Optional: debug model load only. Skip this cell — run the pipeline cell directly.
# On Colab, 3B often crashes (~12 GB RAM limit); pipeline auto-uses 1B unless
# VERITAS_FORCE_MODEL=1.
import os
import torch
import psutil

assert torch.cuda.is_available(), "Enable T4 GPU + restart runtime before this cell."
os.environ['VERITAS_DEVICE'] = 'cuda'
# os.environ['VERITAS_FORCE_MODEL'] = '1'  # uncomment to try 3B (may crash)
# os.environ['VERITAS_MODEL'] = 'meta-llama/Llama-3.2-3B-Instruct'

from interp.activation_cache import load_model, log_device_choice
log_device_choice()
print(f"System RAM before load: {psutil.virtual_memory().used / 1e9:.1f} GB")
model = load_model()
print('Loaded:', model.cfg.model_name)
print('param device:', next(model.parameters()).device)
print(f"GPU allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"System RAM after load: {psutil.virtual_memory().used / 1e9:.1f} GB")

In [ ]:
# Full pipeline, in-process. Wait for '=== Done ==='.
import os
os.environ['VERITAS_DEVICE'] = 'cuda'
# os.environ['VERITAS_SMOKE_N'] = '2'  # quick smoke (2 trajectories)
# os.environ['VERITAS_FORCE_MODEL'] = '1'  # try 3B on Colab (may crash)
# os.environ['VERITAS_TRAJECTORY_SOURCE'] = 'replay'  # legacy foreign replay

import run_pipeline
run_pipeline.main()

In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display

for p in ['results/probe_results.json', 'results/accuracy_by_position.png',
          'results/poc_summary.md', 'results/skepticism_report.md']:
    assert Path(p).exists(), f'Missing {p} - run the pipeline cell to completion'

for img in ['results/accuracy_by_position.png', 'results/accuracy_by_layer.png', 'results/probe_heatmap.png']:
    display(Image(img))
display(Markdown(open('results/poc_summary.md').read()))